<a href="https://colab.research.google.com/github/PRATHAMESHK24/ML-PARCTICAL.ML/blob/main/Practical9_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# practical9_svm.py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=400, noise=0.22, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "Linear SVM": SVC(kernel="linear", C=1),
    "RBF SVM": SVC(kernel="rbf", C=1, gamma="scale"),
    "Polynomial SVM": SVC(kernel="poly", degree=3, C=1, gamma="scale")
}

# Train and visualize each SVM
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, svm) in zip(axes, models.items()):
    model = make_pipeline(StandardScaler(), svm)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(name, "accuracy:", accuracy_score(y_test, pred))

    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 250),
        np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 250)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    decision = model.decision_function(grid).reshape(xx.shape)

    ax.contourf(xx, yy, decision > 0, alpha=0.2)
    ax.contour(
        xx, yy, decision,
        levels=[-1, 0, 1],
        linestyles=["--", "-", "--"]
    )

    ax.scatter(X_train[:, 0], X_train[:, 1],
               c=y_train, edgecolors="black", s=20)

    # Recover support vectors in original feature coordinates
    scaled_svm = model.named_steps["svc"]
    scaler = model.named_steps["standardscaler"]
    support_original = scaler.inverse_transform(
        scaled_svm.support_vectors_
    )
    ax.scatter(
        support_original[:, 0], support_original[:, 1],
        s=80, facecolors="none", edgecolors="red",
        label="Support vectors"
    )

    ax.set_title(name)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.legend()

plt.tight_layout()
plt.show()

# Hyperparameter tuning for the RBF kernel
pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf"))

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": [0.01, 0.1, 1, 10]
}

search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy"
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best cross-validation accuracy:", search.best_score_)
print("Test accuracy:",
      accuracy_score(y_test, search.predict(X_test)))
